# Data Download Pipeline

Downloads all files required by `Batch_TCA_Pipeline.ipynb` from the Zhong et al. (2025) Figshare dataset 
([article 28811129](https://janelia.figshare.com/articles/dataset/Unsupervised_pretraining_in_biological_neural_network/28811129/2)).

**Cohorts:**
- 3 supervised mice (VR2, TX60, TX108) — water-restricted, trained on leaf vs circle discrimination with reward
- 3 unsupervised mice (TX105, TX123, TX83) — exposed to the same corridors without reward

**Per mouse:** two sessions (before and after learning), each requiring:
- `{session}_SVD_dec.npy` — SVD-decomposed neural data
- `{base}_trans.npz` — retinotopy / cortical area labels

**Per cohort × phase:** one shared behavioral file (a dict keyed by session ID, containing all mice in that cohort):
- `Beh_{cohort}_train1_{phase}.npy`
- e.g. `Beh_sup_train1_before_learning.npy` contains keys for VR2, TX60, TX108 (and TX109)

Total: 28 files (~1.9 GB).

In [6]:
from pathlib import Path
import os
import requests

## Setup data directory

In [7]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Data directory: {DATA_DIR}")

Data directory: /Users/johnmadrid/GitHub/isp-unsupervised-learning/data


## Define cohorts, mice, and sessions

Each mouse has a *before learning* and *after learning* session ID.
The Batch TCA pipeline loads three file types per session:

| File | Pattern | Scope |
|------|---------|-------|
| Neural (SVD) | `{session}_SVD_dec.npy` | one file per session |
| Area labels | `{session[:-1]}trans.npz` | one file per session |
| Behavior | `Beh_{cohort}_train1_{phase}.npy` | one file per cohort × phase (dict keyed by session ID, shared across all mice in that cohort) |

In [8]:
SUPERVISED_MICE = {
    "VR2":   {"before": "VR2_2021_03_20_1",   "after": "VR2_2021_04_06_1"},
    "TX60":  {"before": "TX60_2021_04_10_1",  "after": "TX60_2021_05_04_1"},
    "TX108": {"before": "TX108_2023_03_13_1", "after": "TX108_2023_03_22_1"},
}

UNSUPERVISED_MICE = {
    "TX105": {"before": "TX105_2022_10_08_2", "after": "TX105_2022_10_19_2"},
    "TX123": {"before": "TX123_2023_12_21_1", "after": "TX123_2024_01_02_1"},
    "TX83":  {"before": "TX83_2022_08_17_1",  "after": "TX83_2022_08_29_1"},
}

print(f"Supervised mice:   {list(SUPERVISED_MICE.keys())}")
print(f"Unsupervised mice: {list(UNSUPERVISED_MICE.keys())}")

Supervised mice:   ['VR2', 'TX60', 'TX108']
Unsupervised mice: ['TX105', 'TX123', 'TX83']


## Build file manifest

Map every required filename to its Figshare file ID.  
IDs come from the Figshare API for [article 28811129 v2](https://janelia.figshare.com/articles/dataset/Unsupervised_pretraining_in_biological_neural_network/28811129/2).

In [9]:
# Figshare file ID for each file we need.
# Neural + area files are per-session; behavior files are per-cohort.

FIGSHARE_IDS = {
    # ---- Behavioral data (shared per cohort x phase) ----
    "Beh_sup_train1_before_learning.npy":   54183863,
    "Beh_sup_train1_after_learning.npy":    54183860,
    "Beh_unsup_train1_before_learning.npy": 54183914,
    "Beh_unsup_train1_after_learning.npy":  54183917,

    # ---- Supervised Mouse 1: VR2 ----
    "VR2_2021_03_20_1_SVD_dec.npy": 54866354,
    "VR2_2021_03_20_trans.npz":     54184211,
    "VR2_2021_04_06_1_SVD_dec.npy": 54866333,
    "VR2_2021_04_06_trans.npz":     54184214,

    # ---- Supervised Mouse 2: TX60 ----
    "TX60_2021_04_10_1_SVD_dec.npy": 54866198,
    "TX60_2021_04_10_trans.npz":     54184145,
    "TX60_2021_05_04_1_SVD_dec.npy": 54866216,
    "TX60_2021_05_04_trans.npz":     54184148,

    # ---- Supervised Mouse 3: TX108 ----
    "TX108_2023_03_13_1_SVD_dec.npy": 54866285,
    "TX108_2023_03_13_trans.npz":     54184049,
    "TX108_2023_03_22_1_SVD_dec.npy": 54866273,
    "TX108_2023_03_22_trans.npz":     54184052,

    # ---- Unsupervised Mouse 1: TX105 ----
    "TX105_2022_10_08_2_SVD_dec.npy": 54866237,
    "TX105_2022_10_08_trans.npz":     54184028,
    "TX105_2022_10_19_2_SVD_dec.npy": 54866150,
    "TX105_2022_10_19_trans.npz":     54184031,

    # ---- Unsupervised Mouse 2: TX123 ----
    "TX123_2023_12_21_1_SVD_dec.npy": 54866060,
    "TX123_2023_12_21_trans.npz":     54184106,
    "TX123_2024_01_02_1_SVD_dec.npy": 54866186,
    "TX123_2024_01_02_trans.npz":     54184112,

    # ---- Unsupervised Mouse 3: TX83 ----
    "TX83_2022_08_17_1_SVD_dec.npy": 54866165,
    "TX83_2022_08_17_trans.npz":     54184157,
    "TX83_2022_08_29_1_SVD_dec.npy": 54866252,
    "TX83_2022_08_29_trans.npz":     54184184,
}

print(f"Total files in manifest: {len(FIGSHARE_IDS)}")

Total files in manifest: 28


## Cross-check manifest against cohort definitions

Verify that every session listed in the cohort dicts has its corresponding `_SVD_dec.npy`,
`_trans.npz`, and `Beh_*.npy` in the manifest.

In [10]:
def expected_files_for_session(session_id, cohort, phase):
    """Return the three filenames the Batch TCA pipeline needs for one session."""
    svd  = f"{session_id}_SVD_dec.npy"
    trans = f"{session_id[:-1]}trans.npz"
    beh  = f"Beh_{cohort}_train1_{phase}_learning.npy"
    return [svd, trans, beh]

missing = []
for cohort_tag, mice in [("sup", SUPERVISED_MICE), ("unsup", UNSUPERVISED_MICE)]:
    for mouse, sessions in mice.items():
        for phase, session_id in sessions.items():
            for fn in expected_files_for_session(session_id, cohort_tag, phase):
                if fn not in FIGSHARE_IDS:
                    missing.append(fn)

if missing:
    print(f"MISSING from manifest ({len(missing)} files):")
    for fn in missing:
        print(f"  {fn}")
else:
    print("Manifest is complete: all sessions have SVD_dec, trans, and Beh files.")

Manifest is complete: all sessions have SVD_dec, trans, and Beh files.


## Download files from Figshare

In [11]:
FIGSHARE_DOWNLOAD_URL = "https://api.figshare.com/v2/file/download"

def download_file(filename, file_id, dest_dir, chunk_size=8 * 1024 * 1024):
    """Stream-download a single file from Figshare. Skip if it already exists."""
    dest = dest_dir / filename
    if dest.exists():
        size_mb = dest.stat().st_size / (1024 * 1024)
        print(f"  [skip] {filename} already exists ({size_mb:.1f} MB)")
        return False

    url = f"{FIGSHARE_DOWNLOAD_URL}/{file_id}"
    resp = requests.get(url, stream=True)
    resp.raise_for_status()

    total = int(resp.headers.get("Content-Length", 0))
    downloaded = 0
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                pct = downloaded / total * 100
                print(f"\r  [download] {filename}  {downloaded / 1e6:.0f}/{total / 1e6:.0f} MB ({pct:.0f}%)", end="", flush=True)

    size_mb = dest.stat().st_size / (1024 * 1024)
    print(f"\r  [done] {filename}  ({size_mb:.1f} MB)" + " " * 20)
    return True

In [12]:
downloaded_count = 0
skipped_count = 0

for filename, file_id in FIGSHARE_IDS.items():
    was_downloaded = download_file(filename, file_id, DATA_DIR)
    if was_downloaded:
        downloaded_count += 1
    else:
        skipped_count += 1

print(f"\nDownloaded {downloaded_count} files, skipped {skipped_count} (already present).")

  [done] Beh_sup_train1_before_learning.npy  (118.8 MB)                    
  [done] Beh_sup_train1_after_learning.npy  (108.1 MB)                    
  [done] Beh_unsup_train1_before_learning.npy  (293.9 MB)                    
  [done] Beh_unsup_train1_after_learning.npy  (314.5 MB)                    
  [done] VR2_2021_03_20_1_SVD_dec.npy  (161.4 MB)                    
  [done] VR2_2021_03_20_trans.npz  (2.8 MB)                    
  [done] VR2_2021_04_06_1_SVD_dec.npy  (135.5 MB)                    
  [done] VR2_2021_04_06_trans.npz  (2.4 MB)                    
  [done] TX60_2021_04_10_1_SVD_dec.npy  (110.2 MB)                    
  [done] TX60_2021_04_10_trans.npz  (1.7 MB)                    
  [done] TX60_2021_05_04_1_SVD_dec.npy  (114.4 MB)                    
  [done] TX60_2021_05_04_trans.npz  (2.0 MB)                    
  [done] TX108_2023_03_13_1_SVD_dec.npy  (165.5 MB)                    
  [done] TX108_2023_03_13_trans.npz  (2.9 MB)                    
  [done] TX108_2

## Verification

Check that every required file exists in `data/` and that behavioral files contain the expected session keys.

In [13]:
import numpy as np


def file_status(path):
    """Return (exists, size_mb) for a path."""
    if path.exists():
        return True, path.stat().st_size / (1024 * 1024)
    return False, 0.0


def file_tag(fn):
    if "SVD_dec" in fn:
        return "SVD_dec.npy"
    if "trans" in fn:
        return "trans.npz"
    return fn


print("=" * 80)
print("DATA VERIFICATION — file existence")
print("=" * 80)

all_ok = True
seen_sizes = {}

for cohort_label, cohort_tag, mice in [
    ("Supervised",   "sup",   SUPERVISED_MICE),
    ("Unsupervised", "unsup", UNSUPERVISED_MICE),
]:
    print(f"\nCohort: {cohort_label} ({len(mice)} mice)")
    print("-" * 60)

    for mouse, sessions in mice.items():
        print(f"\n  Mouse {mouse}:")
        for phase in ["before", "after"]:
            session_id = sessions[phase]
            svd_fn  = f"{session_id}_SVD_dec.npy"
            trans_fn = f"{session_id[:-1]}trans.npz"
            print(f"    {phase.capitalize()} learning ({session_id}):")
            for fn in [svd_fn, trans_fn]:
                ok, size_mb = file_status(DATA_DIR / fn)
                status = "OK" if ok else "MISSING"
                print(f"      {file_tag(fn):18s}  [{status:>7s}]  {size_mb:>7.1f} MB")
                if not ok:
                    all_ok = False
                seen_sizes[fn] = size_mb

print(f"\nBehavioral data (one file per cohort x phase, dict keyed by session ID)")
print("-" * 60)
beh_files = sorted(fn for fn in FIGSHARE_IDS if fn.startswith("Beh_"))
for fn in beh_files:
    ok, size_mb = file_status(DATA_DIR / fn)
    status = "OK" if ok else "MISSING"
    print(f"  {fn:45s}  [{status:>7s}]  {size_mb:>7.1f} MB")
    if not ok:
        all_ok = False
    seen_sizes[fn] = size_mb

total_size = sum(seen_sizes.values())

print("\n" + "=" * 80)
n_sup = len(SUPERVISED_MICE)
n_unsup = len(UNSUPERVISED_MICE)
print(f"Total: {n_sup} supervised + {n_unsup} unsupervised mice")
print(f"       {n_sup + n_unsup} mice x 2 sessions (before + after) = {(n_sup + n_unsup) * 2} session-pairs")
print(f"       {len(FIGSHARE_IDS)} files, {total_size:.0f} MB on disk")

if all_ok:
    print("\nAll files present on disk.")
else:
    print("\nSome files are MISSING. Re-run the download cell above.")
print("=" * 80)

DATA VERIFICATION — file existence

Cohort: Supervised (3 mice)
------------------------------------------------------------

  Mouse VR2:
    Before learning (VR2_2021_03_20_1):
      SVD_dec.npy         [     OK]    161.4 MB
      trans.npz           [     OK]      2.8 MB
    After learning (VR2_2021_04_06_1):
      SVD_dec.npy         [     OK]    135.5 MB
      trans.npz           [     OK]      2.4 MB

  Mouse TX60:
    Before learning (TX60_2021_04_10_1):
      SVD_dec.npy         [     OK]    110.2 MB
      trans.npz           [     OK]      1.7 MB
    After learning (TX60_2021_05_04_1):
      SVD_dec.npy         [     OK]    114.4 MB
      trans.npz           [     OK]      2.0 MB

  Mouse TX108:
    Before learning (TX108_2023_03_13_1):
      SVD_dec.npy         [     OK]    165.5 MB
      trans.npz           [     OK]      2.9 MB
    After learning (TX108_2023_03_22_1):
      SVD_dec.npy         [     OK]    145.0 MB
      trans.npz           [     OK]      2.6 MB

Cohort: Un

### Verify behavioral file contents

Each `Beh_*.npy` is a dict keyed by session ID. The Batch TCA pipeline does
`beh_all[target_file]` to select a specific mouse's behavior.
Verify that every session ID we need is present as a key.

In [14]:
BEH_MAP = {
    ("sup",   "before"): "Beh_sup_train1_before_learning.npy",
    ("sup",   "after"):  "Beh_sup_train1_after_learning.npy",
    ("unsup", "before"): "Beh_unsup_train1_before_learning.npy",
    ("unsup", "after"):  "Beh_unsup_train1_after_learning.npy",
}

print("=" * 80)
print("DATA VERIFICATION — behavioral file contents")
print("=" * 80)

beh_ok = True
for cohort_tag, mice in [("sup", SUPERVISED_MICE), ("unsup", UNSUPERVISED_MICE)]:
    for phase in ["before", "after"]:
        beh_fn = BEH_MAP[(cohort_tag, phase)]
        beh_path = DATA_DIR / beh_fn
        if not beh_path.exists():
            print(f"\n  {beh_fn}: FILE NOT FOUND — skipping key check")
            beh_ok = False
            continue

        beh_dict = np.load(beh_path, allow_pickle=True).item()
        available_keys = set(beh_dict.keys())
        print(f"\n  {beh_fn}")
        print(f"    Keys in file: {sorted(available_keys)}")

        for mouse, sessions in mice.items():
            session_id = sessions[phase]
            if session_id in available_keys:
                print(f"    {session_id:30s}  [  FOUND]")
            else:
                print(f"    {session_id:30s}  [MISSING]")
                beh_ok = False

if beh_ok:
    print(f"\nAll session IDs found in their behavioral files.")
else:
    print(f"\nSome session IDs are MISSING from behavioral files!")

print("=" * 80)

if all_ok and beh_ok:
    print("\nReady for Batch_TCA_Pipeline.")
else:
    print("\nIssues detected — see above.")

DATA VERIFICATION — behavioral file contents

  Beh_sup_train1_before_learning.npy
    Keys in file: ['TX108_2023_03_13_1', 'TX109_2023_03_27_1', 'TX60_2021_04_10_1', 'VR2_2021_03_20_1']
    VR2_2021_03_20_1                [  FOUND]
    TX60_2021_04_10_1               [  FOUND]
    TX108_2023_03_13_1              [  FOUND]

  Beh_sup_train1_after_learning.npy
    Keys in file: ['TX108_2023_03_22_1', 'TX109_2023_04_07_1', 'TX60_2021_05_04_1', 'VR2_2021_04_06_1']
    VR2_2021_04_06_1                [  FOUND]
    TX60_2021_05_04_1               [  FOUND]
    TX108_2023_03_22_1              [  FOUND]

  Beh_unsup_train1_before_learning.npy
    Keys in file: ['DR10_2022_07_12_1', 'DR15_2022_10_09_1', 'TX104_2022_10_12_1', 'TX105_2022_10_08_2', 'TX119_2023_12_14_1', 'TX123_2023_12_21_1', 'TX83_2022_08_17_1', 'TX85_2022_06_14_1', 'TX88_2022_06_13_2']
    TX105_2022_10_08_2              [  FOUND]
    TX123_2023_12_21_1              [  FOUND]
    TX83_2022_08_17_1               [  FOUND]

  Beh